In [1]:
import os
import numpy as np
import math
from scipy import stats
from load import load_seg_to_bin, load_seg_pair

ref_genome_folder = 'Data/ref_genome/hg19'
exclude_chr = ['chrM', 'chrY','chrMT']
resolution = 50000

# load ref genome length info
chr_length = dict()
with open(ref_genome_folder + '/hg19.len', 'r') as f:
    lines = f.read().splitlines()
    for line in lines:
        chr_name , len_ = line.split('\t')[:2]
        if chr_name in exclude_chr:
            continue
        chr_length[chr_name] = int(len_)

# calculate bin number in each chr
chr_bin_number = dict()
for key in chr_length.keys():
    chr_bin_number[key] = math.floor(chr_length[key] / resolution) + 1


In [18]:
def calculate_precision_racall(trg_seg_file, 
                                groundtruth_file = 'CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt', 
                                copy_ratio_threshold : float = 0.3, 
                                iou_threshold : float = 0.50):
    gt_seg_pair_dict = load_seg_pair(groundtruth_file, threshold = copy_ratio_threshold, has_offset = True)
    trg_seg_pair_dicr = load_seg_pair(trg_seg_file, threshold = copy_ratio_threshold)

    # report the number of ground truth and targer
    num_gt, num_trg = 0, 0
    for chr_ in gt_seg_pair_dict.keys():
        num_gt = num_gt + len(gt_seg_pair_dict[chr_])
        num_trg = num_trg + len(trg_seg_pair_dicr[chr_])
    print(f'Number of ground truth : {num_gt}')
    print(f'Number of target : {num_trg}')

    # generate seg that overlap over threshold
    def getIntercept(a, b):
        return max(0, min(a[1], b[1]) - max(a[0], b[0]))
    def getUnion(a, b):
        return max(a[1], b[1]) - min(a[0], b[0])
    def IOU(a, b):
        overlap_size = getIntercept(a,b)
        IOU_frac = getIntercept(a,b) / getUnion(a,b)
        return IOU_frac, overlap_size
    hit_pair_dict = dict()
    overlap_sizes = list()
    overlap_fracs = list()
    for key in chr_bin_number.keys():
        hit_pair_dict[key] = list()
    for chr_ in trg_seg_pair_dicr.keys():
        for pair in trg_seg_pair_dicr[chr_]:
            for gt_pair in gt_seg_pair_dict[chr_]:
                overlap_frac, overlap_size = IOU(pair, gt_pair)
                if overlap_frac >= iou_threshold:
                    hit_pair_dict[chr_].append(pair)
                    overlap_sizes.append(overlap_size)
                    overlap_fracs.append(overlap_frac)
    print(f'Number of hits : {len(overlap_fracs)}')

    # precision and recall
    num_gt_pair = 0
    for chr_ in gt_seg_pair_dict.keys():
        num_gt_pair = num_gt_pair + len(gt_seg_pair_dict[chr_]) 
    num_trg_pair = 0
    for chr_ in trg_seg_pair_dicr.keys():
        num_trg_pair = num_trg_pair + len(trg_seg_pair_dicr[chr_])
    num_hit_pair = 0 
    for chr_ in hit_pair_dict.keys():
        num_hit_pair = num_hit_pair + len(hit_pair_dict[chr_])
    precision = round(num_hit_pair / num_trg_pair,3)
    recall = round(num_hit_pair / num_gt_pair,3)
    print(f'precision : {precision}\nrecall : {recall}\n')
    
    # hit size by length
    print("hits by size : ")
    print("\t < 1Mb : ", len([i for i in overlap_sizes if i < 1000000]))
    print("\t1Mb - 2Mb : ", len([i for i in overlap_sizes if 1000000 <= i < 2000000]))
    print("\t2Mb - 5Mb : ", len([i for i in overlap_sizes if 2000000 <= i < 5000000]))
    print("\t5Mb - 10Mb : ", len([i for i in overlap_sizes if 5000000 <= i < 10000000]))
    print("\t10Mb - 20Mb : ", len([i for i in overlap_sizes if 10000000 <= i < 20000000]))
    print("\t20Mb - 50Mb : ", len([i for i in overlap_sizes if 20000000 <= i < 50000000]))
    print("\t>= 50Mb : ", len([i for i in overlap_sizes if i >= 50000000]))

    # avg. hit size and hit frac
    avg_overlap_size = round(np.mean(overlap_sizes),3)
    print(f'average hit size : {avg_overlap_size}')
    avg_overlap_frac = round(np.mean(overlap_fracs),3)
    print(f'average hit fraction : {avg_overlap_frac}')

In [19]:
calculate_precision_racall('CNV/K562_HIC_CNV/HiNTcnv_OUTPUT_K562_HindIII/segmentation/K562_CNV_segments.txt')

Number of ground truth : 74
Number of target : 24
Number of hits : 18
precision : 0.75
recall : 0.243

hits by size : 
	 < 1Mb :  0
	1Mb - 2Mb :  0
	2Mb - 5Mb :  1
	5Mb - 10Mb :  1
	10Mb - 20Mb :  1
	20Mb - 50Mb :  8
	>= 50Mb :  7
average hit size : 43991001.111
average hit fraction : 0.855


In [14]:
gt_seg_pair_dict = load_seg_pair('CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt', has_p_value=False, threshold = 0.3, has_offset=True)
trg_seg_pair_dicr = load_seg_pair('CNV/K562_HIC_CNV/HiNTcnv_OUTPUT_K562/segmentation/K562_CNV_segments.txt', has_p_value=True, threshold = 0.3)


In [14]:
num = 0
for chr_ in gt_seg_pair_dict.keys():
    for pair in gt_seg_pair_dict[chr_]:
        num = num +1
print('gt_num:', num)
num = 0
for chr_ in trg_seg_pair_dicr.keys():
    for pair in trg_seg_pair_dicr[chr_]:
        num = num +1
print('trg_num:', num)
num = 0
for chr_ in hit_pair_dict.keys():
    for pair in hit_pair_dict[chr_]:
        num = num +1
print('hit_num:', num)

gt_num: 74
trg_num: 24
hit_num: 18


In [70]:
size_interval_list = [[0,999999], [1000000, 1999999], [2000000, 4999999], [5000000, 9999999], [10000000, 19999999], [20000000, 49999999], [50000000, np.inf]]
hit_by_size = dict()
for chr_ in hit_pair_dict.keys():
    for pair in hit_pair_dict[chr_]:
        size = pair[1] - pair[0]
        for size_interval in size_interval_list:
            if size >= size_interval[0] and size <= size_interval[1]:
                hit_by_size[f'{size_interval[0]} - {size_interval[1]}'] = hit_by_size.get(f'{size_interval[0]} - {size_interval[1]}', 0) + 1
gt_by_size = dict()
for chr_ in gt_seg_pair_dict.keys():
    for pair in gt_seg_pair_dict[chr_]:
        size = pair[1] - pair[0]
        for size_interval in size_interval_list:
            if size >= size_interval[0] and size <= size_interval[1]:
                gt_by_size[f'{size_interval[0]} - {size_interval[1]}'] = gt_by_size.get(f'{size_interval[0]} - {size_interval[1]}', 0) + 1
            
            

In [61]:
hit_by_size

{'50000000 - inf': 7,
 '20000000 - 49999999': 9,
 '10000000 - 19999999': 1,
 '2000000 - 4999999': 1}

In [3]:
from convert import convert_seg_to_bed
convert_seg_to_bed('CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt', output_bed_path = 'wgs.bed', has_offset=True)
convert_seg_to_bed('CNV/K562_HIC_CNV/HiNTcnv_OUTPUT_K562/segmentation/K562_CNV_segments.txt', output_bed_path = 'cnv.bed')

In [4]:
os.system('bedtools intersect -a wgs.bed -b cnv.bed -f 0.5 -r -wo > hit.txt')

0